# 🤖 하나봇 파인튜닝 (Google Colab)

Unsloth + Qwen2.5-3B-Instruct (4-bit)로 하나봇 모델을 파인튜닝합니다.

## 1️⃣ GPU 확인

In [ ]:
import torch
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'메모리: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f}GB')

## 2️⃣ Unsloth 설치 (3-5분 소요)

In [ ]:
# [1-3] Unsloth가 맞는 버전의 transformers · trl 등을 함께 설치합니다
!pip install unsloth -q
print('✓ 설치 완료')

## 3️⃣ 모델 로드 (Qwen2.5-3B-Instruct-bnb-4bit)

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='unsloth/Qwen2.5-3B-Instruct-bnb-4bit',
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
)
print('✓ 모델 로드 완료')

## 4️⃣ LoRA 설정 (r=16, lora_alpha=16)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0.05,
    bias='none',
    use_gradient_checkpointing=True,
    use_rslora=True,
)
print('✓ LoRA 적용 완료')

## 5️⃣ 학습 데이터 업로드

In [ ]:
from google.colab import files

print('data/finetune/에서 hanabot_train.jsonl 파일을 선택해주세요')
uploaded = files.upload()
train_file = list(uploaded.keys())[0]
print(f'업로드됨: {train_file}')

## 6️⃣ 데이터 로드 및 형식 확인

In [ ]:
import json
from datasets import Dataset

# 데이터 로드
data = []
with open(train_file, 'r', encoding='utf-8') as f:
    for line in f:
        data.append(json.loads(line))

print(f'총 {len(data)}개 데이터')

# Chat template으로 학습 텍스트 생성
texts = []
for item in data:
    text = tokenizer.apply_chat_template(
        item['conversations'],
        tokenize=False,
        add_generation_prompt=False
    )
    texts.append(text)

dataset = Dataset.from_dict({'text': texts})
print(f'샘플 텍스트: {texts[0][:100]}...')

## 7️⃣ 학습 설정 (T4 GPU는 fp16)

In [ ]:
# [1-3] 최신 trl은 학습 설정을 SFTConfig에 넣습니다
from trl import SFTTrainer, SFTConfig
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    args=SFTConfig(
        dataset_text_field='text',
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        num_train_epochs=2,
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),   # T4 → fp16
        bf16=is_bfloat16_supported(),
        logging_steps=5,
        optim='adamw_8bit',
        output_dir='outputs',
        seed=42,
        report_to='none',
    ),
)
print('✓ 학습 설정 완료')

## 8️⃣ 학습 실행 (T4 기준 약 10~30분)

In [ ]:
# [1-3] 학습 시작 · 아래에 loss가 5단계마다 찍힙니다
trainer_stats = trainer.train()
print('✓ 학습 완료')

## 9️⃣ Loss 그래프 (학습 과정 시각화)

In [ ]:
# [1-3] 학습 기록으로 loss 그래프 그리기 (내려가면 잘 배우는 중)
import matplotlib.pyplot as plt

logs = trainer.state.log_history
steps = [l['step'] for l in logs if 'loss' in l]
losses = [l['loss'] for l in logs if 'loss' in l]

plt.figure(figsize=(10, 4))
plt.plot(steps, losses, marker='o')
plt.xlabel('Step')
plt.ylabel('Loss')
plt.title('Learning Curve')
plt.grid()
plt.show()

## 🔟 평가 (학습한 모델의 답이 하나봇 형식인지 확인)

In [ ]:
# [1-3] 평가 데이터 업로드 → 질문 3개를 학습한 모델에게 물어보기
import json
from google.colab import files

print('data/finetune/에서 hanabot_eval.jsonl 파일을 선택해주세요')
uploaded = files.upload()
eval_file = list(uploaded.keys())[0]

eval_data = []
with open(eval_file, 'r', encoding='utf-8') as f:
    for i, line in enumerate(f):
        if i < 3:
            eval_data.append(json.loads(line))

FastLanguageModel.for_inference(model)
for i, item in enumerate(eval_data, 1):
    q = item['conversations'][0]['content']
    a = item['conversations'][1]['content']
    inputs = tokenizer.apply_chat_template(
        [{'role': 'user', 'content': q}],
        tokenize=True, add_generation_prompt=True, return_tensors='pt',
    ).to('cuda')
    out = model.generate(input_ids=inputs, max_new_tokens=150, do_sample=False)
    answer = tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True)
    print(f'\n[샘플 {i}] Q: {q}')
    print(f'모델 답:\n{answer}')
    print(f'정답:\n{a}')

## 1️⃣1️⃣ 모델 내보내기 (GGUF q4_k_m · 약 2GB · 5~10분)

In [ ]:
# [1-3] 폴더(gguf_out)에 만들어진 .gguf 중 q4_k_m 파일을 hanabot_finetuned.gguf 로 복사
import glob, os, shutil

model.save_pretrained_gguf('gguf_out', tokenizer, quantization_method='q4_k_m')

found = glob.glob('gguf_out*/**/*.gguf', recursive=True) + glob.glob('*.gguf')
found = [f for f in found if os.path.basename(f) != 'hanabot_finetuned.gguf']
q4 = [f for f in found if 'q4_k_m' in f.lower()]
src = (q4 or sorted(found, key=os.path.getsize))[-1]
shutil.copy(src, 'hanabot_finetuned.gguf')
print('찾은 파일:', found)
print(f'✓ hanabot_finetuned.gguf ({os.path.getsize("hanabot_finetuned.gguf") / 1e9:.1f}GB)')

## 1️⃣2️⃣ Modelfile 생성 (Ollama 문법 · Qwen 대화 형식)

In [ ]:
# [1-3] Qwen(ChatML) 대화 형식 · 학습할 때와 같은 형식이어야 합니다
modelfile = """FROM ./hanabot_finetuned.gguf
TEMPLATE \"\"\"{{ if .System }}<|im_start|>system
{{ .System }}<|im_end|>
{{ end }}{{ if .Prompt }}<|im_start|>user
{{ .Prompt }}<|im_end|>
{{ end }}<|im_start|>assistant
{{ .Response }}<|im_end|>
\"\"\"
PARAMETER stop "<|im_start|>"
PARAMETER stop "<|im_end|>"
PARAMETER temperature 0.3
SYSTEM "당신은 하나 AI 학원의 도우미 하나봇입니다."
"""

with open('Modelfile', 'w', encoding='utf-8') as f:
    f.write(modelfile)
print(modelfile)
print('✓ Modelfile 생성 완료')

## 1️⃣3️⃣ Google Drive에 저장 (권장)

In [ ]:
# [1-3] 내 Drive의 hanabot_finetune 폴더에 GGUF와 Modelfile 저장
from google.colab import drive
import os, shutil

drive.mount('/content/drive')
drive_path = '/content/drive/MyDrive/hanabot_finetune'
os.makedirs(drive_path, exist_ok=True)
shutil.copy('hanabot_finetuned.gguf', drive_path)
shutil.copy('Modelfile', drive_path)

print('✓ Google Drive에 저장 완료:', drive_path)
print('  → 내 PC로 내려받아 hanabot/models/hanabot-ft/ 에 넣으세요')

## 1️⃣4️⃣ (선택) 브라우저로 바로 내려받기

In [ ]:
# [1-3] 2GB라 끊길 수 있습니다. 끊기면 Drive에서 내려받으세요
from google.colab import files
files.download('Modelfile')
files.download('hanabot_finetuned.gguf')